# Nearly incompressible plane strain — Q4-FI vs Q4-SRI

### A LowLevelFEM.jl benchmark, **without changes to the package core**

This notebook reproduces the **rectangular cantilever benchmark B2** in Borsodi Bálint's
2026 BME TDK thesis, supervised by Dr. Attila Kossa, *Közel inkompresszibilis Q4 síkelem
mixed/hibrid u–p formulációval* (Section 6.4, B2; Section 7.1, Table 7.1).

**What is computed**
- **Q4-FI:** bilinear quadrilaterals, fully integrated linear plane-strain elasticity.
- **Q4-SRI:** fully integrated deviatoric strain energy and **1×1** integrated volumetric energy.
- **Equivalent condensed Q4-UP0 displacement:** by the Malkus–Hughes equivalence established
  in Section 4.6 of the thesis, Q4-SRI and elementwise-constant-pressure Q4-UP0 have
  the same element stiffness on valid bilinear quadrilaterals.
- **Recovered elementwise P0 pressure:** \(p_e=-\kappa\,\overline{\nabla\cdot u}_e\), using
  LowLevelFEM's field differentiation and a local average.

**Not included:** an independent global \(u\)–\(p\) saddle-point solve, pressure inf–sup
stability analysis, bubble enrichment, or an exactly incompressible \(\kappa=\infty\) solve.

Everything needed for FEM assembly and solution comes from **LowLevelFEM.jl**:
its Gmsh mesh helper, field descriptor, weak-form DSL, quadrature, boundary loads,
solver, and field calculus. No user-defined element stiffness or assembly loops.

**Units:** SI; 2D plane-strain quantities are **per unit out-of-plane thickness**.
The actual specimen thickness \(t\) therefore enters through the applied traction.

## 1. Requirements

A Julia kernel and the registered `LowLevelFEM` package (current package API).
Gmsh is included through LowLevelFEM. No plotting package is needed:
the notebook displays small, self-contained SVG figures.

Install the package once in your Julia environment if needed:
`import Pkg; Pkg.add("LowLevelFEM")`.

**Run cells from top to bottom**. The Gmsh model is deliberately kept alive during
the notebook, and the final cleanup cell can be run when finished.

In [11]:
using LowLevelFEM
using LinearAlgebra
using Statistics
using Printf

if gmsh.isInitialized() == 0
    gmsh.initialize()
end

println("LowLevelFEM / Gmsh ready.")

LowLevelFEM / Gmsh ready.


## 2. The same rectangular B2 specimen as in the thesis

- Geometry: \(L=8\,\mathrm{m}\), \(H=1\,\mathrm{m}\), \(t=0.1\,\mathrm{m}\).
- **8×2 Q4** mesh.
- Left edge: \(u_x=u_y=0\).
- Right edge: **total vertical force** \(F_y=-1000\,\mathrm{N}\), uniformly distributed.
- Material: \(E=200\,\mathrm{GPa}\), varying Poisson ratio
  \(\nu\in\{0.30,0.45,0.49,0.499,0.4999\}\).
- Output: arithmetic mean of \(u_y\) at all **right-edge nodes**, normalized
  by the Q4-SRI result at \(\nu=0.30\).

In the 2D weak form, the equivalent right-edge traction is
\(q_y=F_y/(tH)=-10\,000\,\mathrm{N/m^2}\).
The stiffness coefficients are 3D elastic moduli in a *plane-strain* model,
not plane-stress coefficients.

In [12]:
const L = 8.0
const H = 1.0
const thickness = 0.1
const E = 200.0e9
const Fy = -1000.0
const nx = 8
const ny = 2
const nu_values = [0.30, 0.45, 0.49, 0.499, 0.4999]

# The package helper creates "body", "left", "right", "top", and "bottom"
# physical groups and a transfinite quadrilateral mesh.
structured_rect_mesh(; lx=L, ly=H, nx=nx, ny=ny, order=1)

U = Field([Material("body")], type=:VectorField, dim=2, fieldName=:u)
support = [BoundaryCondition("left"; field=U, ux=0.0, uy=0.0)]
traction_y = Fy / (thickness * H)
f = ∫(U ⋅ [0.0, traction_y]; Γ="right")

# Validate the total physical load, per unit out-of-plane thickness.
@assert isapprox(sum(f.a[2:2:end, 1]), Fy / thickness; rtol=1e-10)
@printf("Mesh: %d×%d Q4; right-side traction: %.1f Pa\n", nx, ny, traction_y)

Mesh: 8×2 Q4; right-side traction: -10000.0 Pa


## 3. Deviatoric–volumetric split and selective reduced integration

The 3D deviatoric split must be used even in plane strain (\(\varepsilon_{zz}=0\)).
The engineering-strain convention of `SymGrad(U)` is
\([\varepsilon_{xx},\varepsilon_{yy},\gamma_{xy}]^T\).

\[
\mathbf D_{\mathrm{dev}} =
G\begin{bmatrix}4/3&-2/3&0\\-2/3&4/3&0\\0&0&1\end{bmatrix},
\quad
\mathbf D_{\mathrm{vol}} =
\kappa\begin{bmatrix}1&1&0\\1&1&0\\0&0&0\end{bmatrix}.
\]

\[
\begin{aligned}
\mathbf K_{\mathrm{FI}} &= \int_{\Omega}^{2\times2}
   \mathbf B^T(\mathbf D_{\mathrm{dev}}+\mathbf D_{\mathrm{vol}})\mathbf B\,d\Omega,\\
\mathbf K_{\mathrm{SRI}} &= \int_{\Omega}^{2\times2}
   \mathbf B^T\mathbf D_{\mathrm{dev}}\mathbf B\,d\Omega+
   \int_{\Omega}^{1\times1}
   \mathbf B^T\mathbf D_{\mathrm{vol}}\mathbf B\,d\Omega.
\end{aligned}
\]

Only the integration rule of the volumetric term changes.
On first-order quadrilaterals, `gauss=:full` and `gauss=:reduced`
give respectively **2×2** and **1×1** integration.

In [13]:
"""Return the deviatoric and volumetric plane-strain matrices (engineering shear)."""
function split_plane_strain(E, nu)
    G = E / (2 * (1 + nu))
    kappa = E / (3 * (1 - 2nu))
    Ddev = G * [4/3 -2/3 0.0;
                -2/3 4/3 0.0;
                0.0  0.0  1.0]
    Dvol = kappa * [1.0 1.0 0.0;
                    1.0 1.0 0.0;
                    0.0 0.0 0.0]
    return Ddev, Dvol, kappa
end

"""Assemble the standard FI and selectively reduced Q4 stiffnesses with LowLevelFEM."""
function assemble_fi_sri(U, E, nu)
    Ddev, Dvol, kappa = split_plane_strain(E, nu)
    B = SymGrad(U)
    Kdev = ∫(B ⋅ Ddev ⋅ B; Ω="body", gauss=:full)
    Kvol_fi = ∫(B ⋅ Dvol ⋅ B; Ω="body", gauss=:full)
    Kvol_sri = ∫(B ⋅ Dvol ⋅ B; Ω="body", gauss=:reduced)
    return (fi=Kdev + Kvol_fi,
            sri=Kdev + Kvol_sri,
            kappa=kappa)
end

"""Average the vertical displacement over the nodes of the right boundary."""
function right_edge_mean_uy(u, L)
    tags, xyz, _ = gmsh.model.mesh.getNodes()
    right_tags = Int[
        tags[i] for i in eachindex(tags)
        if abs(xyz[3i - 2] - L) < 1e-9
    ]
    isempty(right_tags) && error("No nodes found on the right boundary.")
    # LowLevelFEM 2D vector DOF ordering: ux(node tag), uy(node tag).
    return mean(u.a[2 .* right_tags, 1])
end

# Independent algebraic check of the constitutive split
# against the standard LowLevelFEM plane-strain material matrix.
Ddev0, Dvol0, _ = split_plane_strain(E, 0.30)
Dstandard = D(:PlaneStrain, Material("body"; E=E, ν=0.30))
@assert isapprox(Ddev0 + Dvol0, Dstandard; rtol=1e-12)
println("PASS: Ddev + Dvol = D(:PlaneStrain).")

PASS: Ddev + Dvol = D(:PlaneStrain).


## 4. Poisson-ratio sweep: volumetric locking

The two matrices are assembled through the **same LowLevelFEM operators**.
The full-integrated split must reproduce the directly assembled standard
plane-strain stiffness, whereas the SRI branch releases the excess
volumetric constraints. No new pressure degrees of freedom are introduced.

In [14]:
results = NamedTuple[]
solutions_sri = Dict{Float64,Any}()

for nu in nu_values
    matrices = assemble_fi_sri(U, E, nu)
    u_fi = solveField(matrices.fi, f; support=support)
    u_sri = solveField(matrices.sri, f; support=support)

    tip_fi = right_edge_mean_uy(u_fi, L)
    tip_sri = right_edge_mean_uy(u_sri, L)

    # For a zero-displacement Dirichlet boundary, the solved field
    # satisfies this work balance even when the constitutive matrix is ill-conditioned.
    @assert isfinite(tip_fi) && isfinite(tip_sri)
    push!(results, (nu=nu, u_fi=tip_fi, u_sri=tip_sri))
    solutions_sri[nu] = u_sri
end

reference = results[1].u_sri
ratio_fi  = [r.u_fi / reference for r in results]
ratio_sri = [r.u_sri / reference for r in results]

println("   nu      Q4-FI / ref.    Q4-SRI / ref.    FI / SRI")
for (i, r) in enumerate(results)
    @printf("  %.4f      %10.5f       %10.5f       %10.5f\n",
            r.nu, ratio_fi[i], ratio_sri[i], r.u_fi / r.u_sri)
end
@printf("\nReference deflection: %.9g m (SRI, nu=0.30)\n", reference)

   nu      Q4-FI / ref.    Q4-SRI / ref.    FI / SRI
  0.3000         0.86445          1.00000          0.86445
  0.4500         0.57225          0.93186          0.61410
  0.4900         0.22431          0.89628          0.25027
  0.4990         0.04199          0.88687          0.04735
  0.4999         0.01837          0.88590          0.02073

Reference deflection: -7.66255597e-05 m (SRI, nu=0.30)


### Thesis cross-check (Table 7.1)

Reference normalized values from the BME work:

| \(\nu\) | Q4-FI | Q4-SRI and condensed Q4-UP0 |
|---:|---:|---:|
| 0.30 | 0.864 | 1.000 |
| 0.45 | 0.572 | 0.932 |
| 0.49 | 0.224 | 0.896 |
| 0.499 | 0.042 | 0.887 |
| 0.4999 | 0.018 | 0.886 |

These are **reference data**, not notebook results. Small differences can arise
from the precise choice of the right-edge average and the implementation of the
nodal traction. The tests below first check robust locking trends and energy
equilibrium; the table comparison is reported separately.

In [15]:
reference_fi  = [0.864, 0.572, 0.224, 0.042, 0.018]
reference_sri = [1.000, 0.932, 0.896, 0.887, 0.886]

@printf("Max. absolute discrepancy vs thesis, FI:  %.5f\n",
        maximum(abs.(ratio_fi .- reference_fi)))
@printf("Max. absolute discrepancy vs thesis, SRI: %.5f\n",
        maximum(abs.(ratio_sri .- reference_sri)))

# These are deliberately formulation-level checks, not a hard assertion
# that the BME table must be reproduced to its last reported decimal.
@assert ratio_fi[end] < 0.10 "Expected strong FI volumetric locking."
@assert ratio_sri[end] > 0.70 "Expected an SRI response without severe locking."
@assert abs(ratio_fi[end]) < abs(ratio_sri[end]) / 5
@assert all(isfinite, ratio_fi) && all(isfinite, ratio_sri)
println("PASS: strong volumetric locking with FI; SRI remains compliant.")

Max. absolute discrepancy vs thesis, FI:  0.00045
Max. absolute discrepancy vs thesis, SRI: 0.00028
PASS: strong volumetric locking with FI; SRI remains compliant.


## 5. Embedded locking curve (no additional plotting dependency)

The SVG is constructed only from the numerical values above. This avoids
making the demonstration depend on Plots.jl or Makie.jl.

In [16]:
"""Display a lightweight inline SVG of the normalized tip displacements."""
function locking_svg(nus, fi, sri)
    width, height = 760, 390
    x0, y0, w, h = 76.0, 56.0, 635.0, 260.0
    xx(i) = x0 + (i - 1) * w / (length(nus) - 1)
    yy(v) = y0 + h * (1 - v / 1.05)

    s = IOBuffer()
    print(s, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' viewBox='0 0 $width $height'>")
    print(s, "<rect width='100%' height='100%' fill='#fff'/>")
    print(s, "<text x='76' y='29' font-family='sans-serif' font-size='20' font-weight='600'>Volumetric locking — Q4 elements</text>")
    for v in 0.0:0.2:1.0
        y = yy(v)
        print(s, @sprintf("<line x1='%.1f' y1='%.1f' x2='%.1f' y2='%.1f' stroke='#e2e8f0'/>",
                         x0, y, x0+w, y))
        print(s, @sprintf("<text x='61' y='%.1f' text-anchor='end' font-family='sans-serif' font-size='12'>%.1f</text>",
                         y+4, v))
    end
    for i in eachindex(nus)
        print(s, @sprintf("<text x='%.1f' y='%.1f' text-anchor='middle' font-family='sans-serif' font-size='12'>%.4g</text>",
                          xx(i), y0+h+23, nus[i]))
    end
    print(s, "<text x='395' y='369' text-anchor='middle' font-family='sans-serif' font-size='13'>Poisson ratio ν</text>")
    print(s, "<text x='18' y='180' transform='rotate(-90 18 180)' font-family='sans-serif' font-size='13'>Normalized tip displacement</text>")
    for (vals, color, label, ly) in ((fi, "#dc5535", "Q4-FI", 73.0),
                                    (sri, "#166f91", "Q4-SRI (= condensed UP0)", 97.0))
        pts = join([@sprintf("%.2f,%.2f", xx(i), yy(vals[i])) for i in eachindex(vals)], " ")
        print(s, "<polyline fill='none' stroke='$color' stroke-width='3' points='$pts'/>")
        for i in eachindex(vals)
            print(s, @sprintf("<circle cx='%.2f' cy='%.2f' r='4.5' fill='%s'/>",
                              xx(i), yy(vals[i]), color))
        end
        print(s, "<text x='470' y='$ly' font-family='sans-serif' font-size='13' fill='$color'>$label</text>")
    end
    print(s, "</svg>")
    display(MIME("text/html"), String(take!(s)))
end

locking_svg(nu_values, ratio_fi, ratio_sri)

Volumetric locking — Q4 elements 0.0 0.2 0.4 0.6 0.8 1.0 0.3 0.45 0.49 0.499 0.4999 Poisson ratio ν Normalized tip displacement Q4-FI Q4-SRI (= condensed UP0)

## 6. Recover elementwise \(P_0\) pressure from the SRI displacement field

For Q4 with an affine rectangular mapping, the divergence of a bilinear
displacement varies linearly inside an element, and its four nodal values
average to the element mean. By thesis Eq. (4.11),

\[
p_e=-\kappa\overline{\operatorname{div}u}_e.
\]

A **minor API detail** matters here: LowLevelFEM's evaluated `∇(...; nabla=:div)`
currently works on a three-component vector field. `expandTo3D(u)` is a
LowLevelFEM helper that adds \(u_z=0\), so it yields the correct
two-dimensional divergence without implementing an element-gradient routine.

Pressure is recovered from displacement; it is **not** an independently solved
global field. It is meaningful for *finite* \(\kappa\).

In [17]:
u_star = solutions_sri[0.4999]
_, _, kappa_star = split_plane_strain(E, 0.4999)

# Only public LowLevelFEM field operations: no custom element derivative.
div_u_elem = ∇(expandTo3D(u_star); nabla=:div)
pressure_p0 = [
    -kappa_star * mean(elem_values[:, 1])
    for elem_values in div_u_elem.A
]

@assert length(pressure_p0) == nx * ny
@assert all(isfinite, pressure_p0)

@printf("Recovered elementwise pressure: min = %.6g Pa, max = %.6g Pa\n",
        minimum(pressure_p0), maximum(pressure_p0))
println("Number of independent recovered P0 values: ", length(pressure_p0))

Recovered elementwise pressure: min = -152371 Pa, max = 152371 Pa
Number of independent recovered P0 values: 16


### A compact P0 pressure map

Colors show **unsmoothed elementwise** pressure, not an averaged nodal stress.
The plot is generated from the mesh element connectivity and the recovered
field values; the element integration and field differentiation remain
LowLevelFEM operations.

In [18]:
"""Return 2D centroids of the elements stored in an elementwise LLFEM field."""
function element_centroids(field)
    tags, xyz, _ = gmsh.model.mesh.getNodes()
    xy = Dict(Int(tag) => (xyz[3i-2], xyz[3i-1]) for (i, tag) in enumerate(tags))
    centers = Tuple{Float64,Float64}[]
    for etag in field.numElem
        _, connectivity, _, _ = gmsh.model.mesh.getElement(etag)
        xx = mean(xy[Int(k)][1] for k in connectivity)
        yy = mean(xy[Int(k)][2] for k in connectivity)
        push!(centers, (xx, yy))
    end
    return centers
end

"""Display an unsmoothed elementwise pressure map for the rectangular mesh."""
function pressure_svg(centers, p, L, H, nx, ny)
    w, h = 760, 240
    x0, y0, wp, hp = 62.0, 68.0, 620.0, 116.0
    pmin, pmax = extrema(p)
    scale = max(pmax - pmin, eps(Float64))
    out = IOBuffer()
    print(out, "<svg xmlns='http://www.w3.org/2000/svg' width='$w' height='$h' viewBox='0 0 $w $h'>")
    print(out, "<rect width='100%' height='100%' fill='white'/>")
    print(out, "<text x='62' y='30' font-family='sans-serif' font-weight='600' font-size='19'>Recovered P0 pressure — ν = 0.4999</text>")
    for (j, (x, y)) in enumerate(centers)
        z = clamp((p[j] - pmin) / scale, 0.0, 1.0)
        # A monotone blue-red palette, deliberately avoiding nodal smoothing.
        red   = round(Int, 42 + 186*z)
        green = round(Int, 96 + 24*(1 - abs(2z-1)))
        blue  = round(Int, 209 - 156*z)
        color = @sprintf("#%02x%02x%02x", red, green, blue)
        left = x0 + (x - L/(2nx)) / L * wp
        top  = y0 + (H - y - H/(2ny)) / H * hp
        print(out, @sprintf(
            "<rect x='%.2f' y='%.2f' width='%.2f' height='%.2f' fill='%s' stroke='white' stroke-width='1.5'/>",
            left, top, wp/nx, hp/ny, color))
    end
    print(out, @sprintf("<text x='62' y='211' font-size='12' font-family='sans-serif'>min: %.4g MPa</text>", pmin/1e6))
    print(out, @sprintf("<text x='430' y='211' font-size='12' font-family='sans-serif'>max: %.4g MPa</text>", pmax/1e6))
    print(out, "</svg>")
    display(MIME("text/html"), String(take!(out)))
end

centers = element_centroids(div_u_elem)
pressure_svg(centers, pressure_p0, L, H, nx, ny)

Recovered P0 pressure — ν = 0.4999 min: -0.1524 MPa max: 0.1524 MPa

## 7. Mechanical balance and a final verification summary

We verify equilibrium for the SRI solution at \(\nu=0.4999\).
For a fully clamped left edge, the sum of vertical reactions should balance
the applied load (per unit thickness). We also verify the discrete
external work equals \(u^T K u\) for zero imposed displacements.

The large \(\kappa/G\) ratio is genuine; these checks validate assembly and
equilibrium, **not** mixed-pressure inf–sup stability.

In [19]:
matrices = assemble_fi_sri(U, E, 0.4999)
u = solutions_sri[0.4999]
rhs = f.a[:, 1]
d = u.a[:, 1]

reactions = matrices.sri.A * d - rhs
vertical_balance = abs(sum(reactions[2:2:end]) + sum(rhs[2:2:end]))
work_external = dot(d, rhs)
work_internal = dot(d, matrices.sri.A * d)
work_error = abs(work_external - work_internal) / max(abs(work_external), 1e-30)

@printf("Reaction balance error (per unit thickness): %.5e N/m\n", vertical_balance)
@printf("Relative energy/work error:                %.5e\n", work_error)

@assert vertical_balance < 1e-5 * abs(Fy / thickness)
@assert work_error < 1e-6
println("PASS: external load, reactions, and discrete energy are consistent.")

Reaction balance error (per unit thickness): 6.24681e-06 N/m
Relative energy/work error:                8.04033e-10
PASS: external load, reactions, and discrete energy are consistent.


## 8. Interpretation and limitations

1. **A core-free LLFEM result:** weak-form assembly of Q4-FI and Q4-SRI directly
   exposes and largely eliminates volumetric locking in this benchmark.
2. **BME mixed formulation connection:** the thesis proves Q4-SRI
   and statically condensed \(Q4/u\!-\!P0\) are algebraically equivalent for
   valid bilinear quadrilaterals, so the SRI displacement is also the
   condensed UP0 displacement for this benchmark.
3. **Pressure recovery:** one elementwise scalar \(p_e\) is obtained without
   extending LLFEM's global degrees of freedom.
4. **Not a universal mixed element:** this does not prove stability of the
   discontinuous Q4/P0 pair, nor support exactly incompressible
   \(\kappa=\infty\) material; additional mixed-field and stabilization
   work would be required.

**Reproducibility:** parameters are in Section 2; the experiment uses the
public LowLevelFEM weak-form and solver API as of November 2026.
For an exact code-version record, run `import Pkg; Pkg.status("LowLevelFEM")`
in your Julia environment and record the package version/commit.

**Suggested figure caption for sharing:**
*Volumetric locking in a nearly incompressible plane-strain Q4 cantilever:
standard full integration versus selective reduced integration, implemented
with the public LowLevelFEM.jl weak-form API without changes to the library core.*

### Optional Gmsh view

With a local GUI, evaluate `showDoFResults(u_star, :uy; name="SRI uy",
visible=true)` and then `openPostProcessor()` to inspect the displacement field.
Do this **before** finalizing Gmsh.

In [20]:
# Optional: uncomment in an interactive desktop session.
# showDoFResults(u_star, :uy; name="Q4-SRI uy (nu=0.4999)", visible=true)
# openPostProcessor()

println("Finished. Gmsh remains initialized so that plots and probes can be explored.")
# Run gmsh.finalize() after all LLFEM results, plots and probes are complete.

Finished. Gmsh remains initialized so that plots and probes can be explored.
